In [139]:
import pandas as pd
import numpy as np
from EssSimulation import EssSimulationModel
import calendar
import copy

In [140]:
exp_name = "estimate830"
month_num = 9
node_name = "route_B_{:02d}".format(month_num)

In [141]:
es_info = {"transform_capacity": 63000,
           "invertband": 0,
           "soc_redundant_ratio": 0,
           "usable_depth": 0.97,
           "charge_loss": 0.92,
           "discharge_loss": 0.95,
           "es_charge_max": 9000,
           "es_charge_min": -9000,
           "es_capacity_max": 18000,
           "es_capacity_min": 0}

In [142]:
ratio_result_list = []
for ratio in range(100, 500, 10):
    demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
    demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
    demand_load_df.set_index('time', inplace=True)

    strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/2stage_ideal_ratio_dod97/schedule_result_fixline_up{ratio}.csv")
    strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
    strategy_df['time'] = pd.to_datetime(strategy_df['time'])
    strategy_df.set_index('time', inplace=True)

    ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
    ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
    ele_price_df.set_index('time', inplace=True)

    simulation_model = EssSimulationModel(es_info)
    es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 4050)
    origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)
    ratio_result_list.append((ratio, origin_balance - opt_balance))
    print(f"突破比例{ratio}%, 收益为{origin_balance - opt_balance}")
    # print(round(origin_balance - opt_balance,0))

突破比例100%, 收益为171530.21712716203
突破比例110%, 收益为187686.1301474888
突破比例120%, 收益为203842.04185443278
突破比例130%, 收益为219997.96287560556
突破比例140%, 收益为236491.60542596038
突破比例150%, 收益为253147.98254100606
突破比例160%, 收益为269804.28626185935
突破比例170%, 收益为285878.13746843114
突破比例180%, 收益为301272.242927636
突破比例190%, 收益为316585.494356486
突破比例200%, 收益为330217.8030604115
突破比例210%, 收益为340048.8912807377
突破比例220%, 收益为347269.0156847704
突破比例230%, 收益为353703.3879705323
突破比例240%, 收益为359900.1872564852
突破比例250%, 收益为365040.6940396922
突破比例260%, 收益为367369.6419809414
突破比例270%, 收益为367186.45454192534
突破比例280%, 收益为366975.8625062313
突破比例290%, 收益为366838.44221451506
突破比例300%, 收益为366700.9779813504
突破比例310%, 收益为366563.55429521296
突破比例320%, 收益为366314.02315074764
突破比例330%, 收益为365934.7792947022
突破比例340%, 收益为365517.8520548241
突破比例350%, 收益为365009.2125117956
突破比例360%, 收益为364499.28041658737
突破比例370%, 收益为363989.31360303704
突破比例380%, 收益为363479.38081424683
突破比例390%, 收益为362969.4131859392
突破比例400%, 收益为362371.18059683405
突破比例410%, 收益为361607.186410

In [143]:
max_ratio_tuple = max(ratio_result_list, key=lambda x: x[1])

In [144]:
max_ratio = max_ratio_tuple[0]
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/2stage_ideal_ratio_dod97/schedule_result_fixline_up{max_ratio}.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 4050)
origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)

In [145]:
print("测算方式一  比例：", {max_ratio}, "收益：", (origin_balance - opt_balance), "收益占比：", (origin_balance - opt_balance) / origin_balance)

测算方式一  比例： {260} 收益： 367369.6419809414 收益占比： 0.07040242168643272


In [146]:
ori_max_demand = demand_load_df["value"].max()
opt_max_demand = total_load_df["total_load"].max()

print("调度后最大需量：", opt_max_demand, "原始最大需量：", ori_max_demand, "需量抬升成本", (opt_max_demand - ori_max_demand) * 38.4)

调度后最大需量： 12851.93 原始最大需量： 10524.0 需量抬升成本 89392.512


In [147]:
mean11 = total_load_df["total_load"].mean() * 1.1
(opt_max_demand - mean11) * 38.4

84511.86538752064